**ASSIGNMENT: Building a Mixture of Experts (MoE) Router**

Install Dependancies

In [1]:
%pip install groq python-dotenv --quiet

Import & Set API Key

In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from groq import Groq

# Securely ask for Groq API key if not already set
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API Key: ")

# Initialize Groq client
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq client initialized successfully.")

Enter your GROQ API Key: ··········
Groq client initialized successfully.


Define Experts (MODEL_CONFIG)

In [3]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": "You are a technical support expert. Provide precise, code-focused debugging help.",
        "model": "llama-3.1-8b-instant",
        "temperature": 0.6
    },

    "billing": {
        "system_prompt": "You are a billing support specialist. Be empathetic and explain refund or payment issues clearly.",
        "model": "llama-3.1-8b-instant",
        "temperature": 0.4
    },

    "general": {
        "system_prompt": "You are a helpful general assistant. Answer casually and clearly.",
        "model": "llama-3.1-8b-instant",
        "temperature": 0.7
    },

    "tool": {
        "system_prompt": None,
        "model": None,
        "temperature": None
    }
}

Router

In [4]:
def route_prompt(user_input):

    routing_prompt = f"""
Classify the query into EXACTLY one category:
[technical, billing, general, tool]

If it asks for real-time data like prices or statistics,
classify as "tool".

Return ONLY the category name.

Query: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You are an intent classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    return response.choices[0].message.content.strip().lower()

Tool Function (Mock Tool)

In [5]:
def get_crypto_price(ticker):
    prices = {
        "bitcoin": "$65,000",
        "ethereum": "$3,500"
    }
    return f"The current mock price of {ticker} is {prices.get(ticker.lower(), 'unknown')}."

Orchestrator

In [6]:
def process_request(user_input):

    category = route_prompt(user_input)
    print("Routed to:", category)

    if category not in MODEL_CONFIG:
        category = "general"

    # Tool handling
    if category == "tool":
        if "bitcoin" in user_input.lower():
            return get_crypto_price("bitcoin")
        elif "ethereum" in user_input.lower():
            return get_crypto_price("ethereum")
        else:
            return "Tool data not available."

    # LLM expert handling
    config = MODEL_CONFIG[category]

    response = client.chat.completions.create(
        model=config["model"],
        temperature=config["temperature"],
        messages=[
            {"role": "system", "content": config["system_prompt"]},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

Test Cases

In [7]:
print(process_request("My Python script is throwing an IndexError on line 5."))
print()
print(process_request("I was charged twice for my subscription this month."))
print()
print(process_request("What are your working hours?"))
print()
print(process_request("What is the current price of Bitcoin?"))

Routed to: technical
To help you debug the issue, I'll need more information about your code. Please provide the following:

1. The exact error message, including the line number and any relevant details.
2. The relevant code snippet (lines 1-10 should be enough) around the line that's throwing the error.
3. Any recent changes you've made to the code.

A common cause of an `IndexError` is trying to access an element in a list or tuple that doesn't exist. For example:

```python
my_list = [1, 2, 3]
print(my_list[5])  # IndexError: list index out of range
```

To fix the issue, you'll need to ensure that the index you're trying to access is within the bounds of the list or tuple.

Here's a step-by-step process to help you debug the issue:

1. **Print the length of the list or tuple**: Before trying to access an element, print the length of the list or tuple to ensure it's what you expect:

    ```python
print(len(my_list))
```

2. **Check the index**: Make sure the index you're trying to